# Etapa 1: Analítica Descriptiva — ¿Qué pasó?
Caso: consorcio que distribuye productos a pulperos.
Preguntas: ¿Qué rutas fallaron? ¿Qué productos se quedaron sin inventario? ¿Qué tan leal es cada pulpero?
Técnicas: frecuencias, porcentajes, medianas y un KPI (índice de lealtad).
**Nota:** los datos son sintéticos, solo para ilustrar la técnica.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option("display.width", 150)

## 1. Datos de ejemplo (en la vida real vendrían de su bodega de datos / BI)

In [ ]:
# --- Entregas ---
pools = {"R1":[1,2,3],"R2":[2,3,4],"R3":[4,5,9],"R4":[5,6,7],
         "R5":[7,8,9],"R6":[8,10,11],"R7":[9,11,12],"R8":[10,12,1]}
region_de = {"R1":"Central","R2":"Central","R3":"Pacífico","R4":"Pacífico",
             "R5":"Caribe","R6":"Caribe","R7":"Norte","R8":"Norte"}
n = 4000
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
rutas = rng.choice(list(pools), n)
choferes = [f"Chofer_{rng.choice(pools[r]):02d}" for r in rutas]
camiones = [f"T-{r}{rng.choice(['a','b'])}" for r in rutas]
mes = fechas.month
p_fallo = 0.08 + np.where(np.array(choferes) == "Chofer_09", 0.30, 0) \
              + np.where((rutas == "R7") & (mes >= 6) & (mes <= 10), 0.15, 0)
fallo = rng.random(n) < p_fallo
entregas = pd.DataFrame({"fecha": fechas, "ruta": rutas, "region": [region_de[r] for r in rutas],
                         "chofer": choferes, "camion": camiones, "fallida": fallo})

# --- Pedidos de pulperos ---
n_pulperos = 150
pulperos = pd.DataFrame({"pulpero_id": [f"P{i:03d}" for i in range(n_pulperos)],
                         "perfil": rng.choice([0.4, 1.0, 2.0], n_pulperos, p=[.4, .4, .2])})
m = 12000
ped = pd.DataFrame({"pulpero_id": rng.choice(pulperos.pulpero_id, m,
                    p=(pulperos.perfil / pulperos.perfil.sum()).values)})
ped = ped.merge(pulperos, on="pulpero_id")
ped["fecha"] = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, m), unit="D")
ped["cantidad"] = rng.poisson(10 * ped.perfil) + 1
ped["monto"] = ped.cantidad * rng.uniform(2.5, 4.5, m)
pedidos = ped.drop(columns="perfil")

# --- Inventario semanal por producto ---
productos = ["Arroz","Frijoles","Aceite","Bloqueador","Gaseosa","Detergente","Leche"]
filas = []
for sem in range(52):
    mes_s = min(12, sem // 4.35 + 1)
    for p in productos:
        base = 100
        est = {"Bloqueador": 2.2 if mes_s <= 3 else 0.8,
               "Gaseosa": 1.8 if mes_s == 12 else 1.0}.get(p, 1.0)
        demanda = rng.poisson(base * est)
        stock = rng.poisson(120)
        filas.append((sem, int(mes_s), p, demanda, stock, demanda > stock))
inventario = pd.DataFrame(filas, columns=["semana","mes","producto","demanda","stock","sin_inventario"])
print(entregas.shape, pedidos.shape, inventario.shape)

## 2. ¿Qué rutas fallaron? (frecuencia y porcentaje)

In [ ]:
rutas_res = (entregas.groupby("ruta")
             .agg(entregas=("fallida","size"), fallidas=("fallida","sum"))
             .assign(pct_fallo=lambda d: (100 * d.fallidas / d.entregas).round(1))
             .sort_values("pct_fallo", ascending=False))
print(rutas_res)
print(f"\nTasa global de fallo: {100 * entregas.fallida.mean():.1f}%")

ax = rutas_res.pct_fallo.plot.bar(color="#c0392b", figsize=(8, 4))
ax.axhline(100 * entregas.fallida.mean(), ls="--", c="gray", label="Promedio global")
ax.set_ylabel("% entregas fallidas"); ax.set_title("Tasa de fallo por ruta"); ax.legend()
plt.tight_layout(); plt.show()

## 3. ¿Qué productos se quedaron sin inventario?

In [ ]:
inv_res = (inventario.groupby("producto")
           .agg(semanas=("semana","size"), semanas_sin_stock=("sin_inventario","sum"),
                demanda_mediana=("demanda","median"))
           .assign(pct_semanas_sin_stock=lambda d: (100 * d.semanas_sin_stock / d.semanas).round(1))
           .sort_values("pct_semanas_sin_stock", ascending=False))
print(inv_res)

ax = inv_res.pct_semanas_sin_stock.plot.barh(color="#2980b9", figsize=(8, 4))
ax.set_xlabel("% de semanas sin inventario"); ax.set_title("Quiebres de inventario por producto")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

## 4. KPI: índice de lealtad del pulpero
Fórmula: `0.4·frecuencia + 0.3·cantidad + 0.3·monto`, cada componente normalizado de 0 a 1 (min-max).
Los pesos son una **definición de negocio**: deben acordarse y mantenerse constantes en todos los reportes.

In [ ]:
W = {"frecuencia": 0.4, "cantidad": 0.3, "monto": 0.3}

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

lealtad = pedidos.groupby("pulpero_id").agg(frecuencia=("fecha","count"),
                                            cantidad=("cantidad","sum"),
                                            monto=("monto","sum"))
for c in W:
    lealtad[c + "_n"] = minmax(lealtad[c])
lealtad["indice_lealtad"] = sum(W[c] * lealtad[c + "_n"] for c in W)
lealtad["segmento"] = pd.qcut(lealtad.indice_lealtad, [0, .5, .8, 1], labels=["Bajo","Medio","Alto"])

print(lealtad.sort_values("indice_lealtad", ascending=False)
      [["frecuencia","cantidad","monto","indice_lealtad","segmento"]].head(10).round(2))
print("\nResumen por segmento (medianas):")
print(lealtad.groupby("segmento", observed=True)[["frecuencia","cantidad","monto"]].median().round(1))

lealtad.indice_lealtad.plot.hist(bins=20, color="#27ae60", figsize=(8, 4))
plt.xlabel("Índice de lealtad"); plt.title("Distribución del índice de lealtad"); plt.tight_layout(); plt.show()

## 5. Mini-reporte (lo que iría a un dashboard de Looker / QuickSight)

In [ ]:
peor = rutas_res.index[0]
print("=== REPORTE DESCRIPTIVO ===")
print(f"Entregas: {len(entregas):,} | Fallidas: {entregas.fallida.sum():,} ({100*entregas.fallida.mean():.1f}%)")
print(f"Ruta con más fallos: {peor} ({rutas_res.pct_fallo.iloc[0]}%)")
print(f"Producto con más quiebres: {inv_res.index[0]} ({inv_res.pct_semanas_sin_stock.iloc[0]}% de semanas)")
print(f"Pulperos de lealtad alta: {(lealtad.segmento == 'Alto').sum()} de {len(lealtad)}")
print("\nOjo: esta etapa dice QUÉ pasó, no POR QUÉ. Eso lo resuelve la analítica diagnóstica.")